In [ ]:
# ===================================================================
# TQ HMM: Regime-Aware Fund Intelligence
# ===================================================================
# Section 0: Setup & Data Load
# Runs server-side (Django shell_plus). Django ORM and Bloomberg con
# object are pre-loaded — no explicit connection setup required.
# ===================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ------------------------------------------------------------------
# Configuration — edit these before running
# ------------------------------------------------------------------
# Good equity fund candidates for S&P regime analysis:
#   RTA   — Purpose Tactical Asset Allocation Fund (multi-asset, Craig Basinger)
#   PDF   — Purpose Core Dividend Fund (equity dividend, BHC)
#   RDE   — Purpose Core Equity Income Fund (equity income, Craig Basinger)
#   PID   — Purpose International Dividend Fund (equity dividend, BHC)
#   SHEQT — Steadyhand Equity Fund (equity growth, Fiera)
#   PBI   — Purpose Best Ideas Fund (equity growth, BHC)
#   PDIV  — Purpose Enhanced Dividend Fund (dividend + options, BHC)
FUND_SYMBOL      = 'RTA'          # FundAccount.symbol on server
BENCHMARK_TICKER = 'SPXT Index'   # S&P 500 Total Return Index
START_DATE       = '2020-01-01'   # inclusive
END_DATE         = '2026-04-22'   # inclusive (update to run date)

# Color constants — Phase 1 baseline (Phase 5 will refine)
COLOR_FUND  = '#1f77b4'   # blue
COLOR_BENCH = '#ff7f0e'   # orange
COLOR_DD    = '#d62728'   # red (drawdown)

# ------------------------------------------------------------------
# Connection context (pre-loaded in server shell_plus)
# ------------------------------------------------------------------
# Django:    FundAccount, Fund models available via ORM
# Bloomberg: con object available (con.bdh, con.ref, con.bds methods)
# ------------------------------------------------------------------

print(f"Setup: {FUND_SYMBOL} vs {BENCHMARK_TICKER}")
print(f"Period: {START_DATE} to {END_DATE}")
print("Note: This notebook runs on server shell_plus.")
print("      Django ORM and Bloomberg con are pre-loaded.")

In [ ]:
# ------------------------------------------------------------------
# Section 0.2: Data Load
# ------------------------------------------------------------------

# --- Fund NAV (Django ORM) ---
# FundAccount.objects.get() raises DoesNotExist if symbol not found
fa = FundAccount.objects.get(symbol=FUND_SYMBOL)
df_nav = fa.primary_fund.get_adj_nav().squeeze()  # ensure Series (get_adj_nav returns DataFrame)

# Filter to requested date range
df_nav = df_nav[
    (df_nav.index >= pd.to_datetime(START_DATE)) &
    (df_nav.index <= pd.to_datetime(END_DATE))
]

print(f"Fund NAV loaded:  {len(df_nav):,} observations")
print(f"  Date range:     {df_nav.index.min().date()} to {df_nav.index.max().date()}")
print(f"  NAV range:      {df_nav.min():.2f} to {df_nav.max():.2f}")
print(f"  Frequency:      {df_nav.index.inferred_freq}")

# --- Benchmark Price (Bloomberg) ---
# con.bdh positional: (ticker, field, start_date, end_date)
df_bench = con.bdh(BENCHMARK_TICKER, 'PX_LAST', START_DATE, END_DATE)
df_bench.columns = ['price']
df_bench_price = df_bench['price']  # Extract as Series

print(f"\nBenchmark loaded: {len(df_bench_price):,} observations")
print(f"  Date range:     {df_bench_price.index.min().date()} to {df_bench_price.index.max().date()}")
print(f"  Price range:    {df_bench_price.min():.2f} to {df_bench_price.max():.2f}")
print(f"  Frequency:      {df_bench_price.index.inferred_freq}")

# --- Frequency check ---
fund_freq  = df_nav.index.inferred_freq
bench_freq = df_bench_price.index.inferred_freq
print(f"\nFrequency check: fund={fund_freq}, benchmark={bench_freq}")
if fund_freq not in ('B', 'D', None) or bench_freq not in ('B', 'D', None):
    print("WARNING: unexpected frequency — verify data sources before proceeding")
else:
    print("Frequency OK — both series appear daily")

In [ ]:
# ------------------------------------------------------------------
# Section 0.3: Log Returns
# Decision D-06: np.log(price_t / price_{t-1}) for both series
# Decision D-07: daily frequency — no resampling
# ------------------------------------------------------------------

# Fund log returns
fund_log_returns = np.log(df_nav / df_nav.shift(1))
fund_log_returns = fund_log_returns.dropna()
fund_log_returns.name = 'fund'

# Benchmark log returns
bench_log_returns = np.log(df_bench_price / df_bench_price.shift(1))
bench_log_returns = bench_log_returns.dropna()
bench_log_returns.name = 'bench'

print(f"Fund log returns:      {len(fund_log_returns):,} observations")
print(f"  Range:               {fund_log_returns.min():.4f} to {fund_log_returns.max():.4f}")
print(f"Benchmark log returns: {len(bench_log_returns):,} observations")
print(f"  Range:               {bench_log_returns.min():.4f} to {bench_log_returns.max():.4f}")

# Sanity check: daily log returns should be between -0.20 and +0.20 for equity funds
if fund_log_returns.abs().max() > 0.20:
    print(f"WARNING: fund has return > 20% in a single day — verify NAV data")
if bench_log_returns.abs().max() > 0.15:
    print(f"WARNING: benchmark has return > 15% in a single day — verify price data")

In [ ]:
# ------------------------------------------------------------------
# Section 0.4: Date Alignment
# Decision D-08: inner join only — no forward-fill, no interpolation
# ------------------------------------------------------------------

# Inner join: keep only dates where both series have valid data
returns_df = pd.DataFrame({
    'fund':  fund_log_returns,
    'bench': bench_log_returns,
})
aligned_returns = returns_df.dropna()  # dropna() = inner join on DatetimeIndex

# --- Alignment statistics ---
n_fund    = len(fund_log_returns)
n_bench   = len(bench_log_returns)
n_aligned = len(aligned_returns)
coverage  = n_aligned / min(n_fund, n_bench) * 100

print(f"Alignment summary:")
print(f"  Fund observations:      {n_fund:,}")
print(f"  Benchmark observations: {n_bench:,}")
print(f"  Aligned observations:   {n_aligned:,}")
print(f"  Coverage:               {coverage:.1f}%")
print(f"  Date range:             {aligned_returns.index.min().date()} to {aligned_returns.index.max().date()}")

if coverage < 90:
    print(f"  WARNING: coverage below 90% — investigate data gaps before proceeding")

# --- Gap detection ---
date_diffs = aligned_returns.index.to_series().diff().dropna()
large_gaps = date_diffs[date_diffs > pd.Timedelta(days=3)]
if len(large_gaps) > 0:
    print(f"\n  Gaps wider than 3 days ({len(large_gaps)} found):")
    for gap_end, gap_size in large_gaps.items():
        gap_start = aligned_returns.index[aligned_returns.index.get_loc(gap_end) - 1]
        print(f"    {gap_start.date()} to {gap_end.date()} ({gap_size.days} days)")
else:
    print(f"\n  No gaps wider than 3 days — date coverage looks clean")

# --- Integrity assertion ---
assert aligned_returns.isnull().sum().sum() == 0, "Unexpected NaN in aligned data — alignment failed"

# --- Output variables (canonical names for downstream phases) ---
fund_aligned  = aligned_returns['fund']   # pd.Series, DatetimeIndex, fund log returns
bench_aligned = aligned_returns['bench']  # pd.Series, DatetimeIndex, benchmark log returns

print(f"\nOutput variables ready:")
print(f"  fund_aligned:  {len(fund_aligned):,} observations, name='{fund_aligned.name}'")
print(f"  bench_aligned: {len(bench_aligned):,} observations, name='{bench_aligned.name}'")
print(f"  Index match:   {(fund_aligned.index == bench_aligned.index).all()}")

## Section 1: Exploratory Data Analysis

Full-history unconditional statistics and charts. This section establishes the baseline against which Phase 3's regime-conditional analysis will be compared. Risk-free rate: 2% annual. Annualization: 252 trading days.

In [ ]:
# ------------------------------------------------------------------
# Section 1.1: Summary Statistics (Full History)
# Decision D-10: annualized mean return, vol, Sharpe, max drawdown
# Annualization: 252 trading days | Risk-free rate: 0.02 (2% annual)
# ------------------------------------------------------------------

def compute_summary_stats(returns_series, risk_free_rate=0.02):
    """
    Annualized statistics from daily log returns.
    Used in Phase 1 (baseline) and Phase 3 (regime-conditional).
    """
    daily_mean = returns_series.mean()
    daily_std  = returns_series.std()

    annual_return = (1 + daily_mean) ** 252 - 1
    annual_vol    = daily_std * np.sqrt(252)
    excess_return = annual_return - risk_free_rate
    sharpe        = excess_return / annual_vol if annual_vol > 0 else 0

    cumret      = (1 + returns_series).cumprod()
    running_max = cumret.expanding().max()
    drawdown    = (cumret - running_max) / running_max
    max_dd      = drawdown.min()

    return {
        'Annual Return':     annual_return,
        'Annual Volatility': annual_vol,
        'Sharpe Ratio':      sharpe,
        'Max Drawdown':      max_dd,
        'Observations':      len(returns_series),
        'Start Date':        returns_series.index.min().strftime('%Y-%m-%d'),
        'End Date':          returns_series.index.max().strftime('%Y-%m-%d'),
    }

fund_stats  = compute_summary_stats(fund_aligned)
bench_stats = compute_summary_stats(bench_aligned)

stats_table = pd.DataFrame({
    'Fund':      fund_stats,
    'Benchmark': bench_stats,
}).T

# Format for display
fmt_table = stats_table.copy()
for col in ['Annual Return', 'Annual Volatility', 'Max Drawdown']:
    fmt_table[col] = stats_table[col].apply(lambda x: f'{x:.2%}')
fmt_table['Sharpe Ratio'] = stats_table['Sharpe Ratio'].apply(lambda x: f'{x:.2f}')
fmt_table['Observations'] = stats_table['Observations'].apply(lambda x: f'{int(x):,}')

print("=== SUMMARY STATISTICS (Full History) ===")
print(fmt_table[['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Max Drawdown', 'Observations', 'Start Date', 'End Date']].to_string())

In [ ]:
# ------------------------------------------------------------------
# Section 1.2: EDA Charts
# Decision D-11: cumulative return, rolling 21-day vol, drawdown
# Decision D-09: figsize=(12, 4), readable but not presentation-quality
# Phase 5 will apply full polish and regime shading.
# ------------------------------------------------------------------

# --- Chart 1: Cumulative Return ---
cumret_fund  = (1 + fund_aligned).cumprod()
cumret_bench = (1 + bench_aligned).cumprod()

fig1, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(cumret_fund.index,  cumret_fund.values,  label=FUND_SYMBOL,      linewidth=2, color=COLOR_FUND)
ax1.plot(cumret_bench.index, cumret_bench.values, label=BENCHMARK_TICKER, linewidth=2, color=COLOR_BENCH)
ax1.set_title('Cumulative Return — Fund vs Benchmark (full history)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Growth of $1')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}x'))
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Chart 2: Rolling 21-Day Volatility (annualized) ---
rolling_vol_fund  = fund_aligned.rolling(window=21).std()  * np.sqrt(252)
rolling_vol_bench = bench_aligned.rolling(window=21).std() * np.sqrt(252)

fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.plot(rolling_vol_fund.index,  rolling_vol_fund.values,  label=FUND_SYMBOL,      linewidth=2, color=COLOR_FUND)
ax2.plot(rolling_vol_bench.index, rolling_vol_bench.values, label=BENCHMARK_TICKER, linewidth=2, color=COLOR_BENCH)
ax2.set_title('Rolling 21-Day Volatility — Annualized')
ax2.set_xlabel('Date')
ax2.set_ylabel('Annualized Volatility')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Chart 3: Fund Drawdown ---
running_max_fund = cumret_fund.expanding().max()
drawdown_fund    = (cumret_fund - running_max_fund) / running_max_fund

max_dd_val  = drawdown_fund.min()
max_dd_date = drawdown_fund.idxmin()

fig3, ax3 = plt.subplots(figsize=(12, 4))
ax3.fill_between(drawdown_fund.index, 0, drawdown_fund.values, color=COLOR_DD, alpha=0.5, label='Drawdown')
ax3.plot(drawdown_fund.index, drawdown_fund.values, color=COLOR_DD, linewidth=1.5)
ax3.axvline(x=max_dd_date, color='black', linestyle='--', linewidth=1, alpha=0.6,
            label=f'Max DD: {max_dd_val:.2%} ({max_dd_date.strftime("%Y-%m-%d")})')
ax3.set_title(f'Fund Drawdown — Max: {max_dd_val:.2%} on {max_dd_date.strftime("%Y-%m-%d")}')
ax3.set_xlabel('Date')
ax3.set_ylabel('Drawdown')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"EDA complete. Max drawdown: {max_dd_val:.2%} on {max_dd_date.strftime('%Y-%m-%d')}")

## 2. Regime Model

Fit a 2-state Gaussian HMM on benchmark returns (`bench_aligned`) to identify latent market regimes. The Viterbi algorithm decodes the most likely sequence of hidden states per date.

In [ ]:
from sklearn.mixture import GaussianMixture

assert bench_aligned.isnull().sum() == 0, "bench_aligned contains NaN -- alignment failed"
assert len(bench_aligned) > 100, "Not enough observations for fitting"

X = bench_aligned.values.reshape(-1, 1)

# 3-component GMM: separates calm/bull, elevated-vol/choppy, and crisis states.
model = GaussianMixture(n_components=3, covariance_type='diag', max_iter=100, tol=1e-4, random_state=0)
model.fit(X)

print(f"Converged: {model.converged_}")
print(f"Log-likelihood: {model.score(X):.4f}")
if not model.converged_:
    print("WARNING: EM did not converge -- consider increasing max_iter")

state_sequence = model.predict(X)

print("\nState distribution:")
for i in range(3):
    n = (state_sequence == i).sum()
    print(f"  State {i}: {n:,} obs  mean={model.means_[i,0]:.5f}  vol={np.sqrt(model.covariances_[i,0]):.5f}")

regime_decoded = pd.Series(state_sequence, index=bench_aligned.index)
regime_decoded.name = 'regime_state'
print(f"\nFirst 20 states: {regime_decoded.head(20).values}")

In [ ]:
# Label states by Sharpe rank: highest = Risk-On, middle = Transition, lowest = Risk-Off.

state_stats = {}
for state_idx in range(3):
    mask = regime_decoded == state_idx
    state_stats[state_idx] = compute_summary_stats(bench_aligned[mask], risk_free_rate=0.02)

sharpe_ranks = sorted(range(3), key=lambda i: state_stats[i]['Sharpe Ratio'], reverse=True)
label_map = {sharpe_ranks[0]: 'Risk-On', sharpe_ranks[1]: 'Transition', sharpe_ranks[2]: 'Risk-Off'}

REGIME_ORDER = ['Risk-On', 'Transition', 'Risk-Off']

print("=== STATE STATISTICS (benchmark returns) ===")
print(f"  {'State':<7}{'Label':<12}{'Return':>10}{'Vol':>10}{'Sharpe':>8}{'MaxDD':>10}{'N':>7}")
for i in range(3):
    s = state_stats[i]
    lbl = label_map[i]
    print(f"  {i:<7}{lbl:<12}{s['Annual Return']:>10.2%}{s['Annual Volatility']:>10.2%}"
          f"{s['Sharpe Ratio']:>8.2f}{s['Max Drawdown']:>10.2%}{int(s['Observations']):>7,}")

print("\nLabel assignment (Sharpe rank):")
for i, lbl in label_map.items():
    print(f"  State {i} -> {lbl}  (Sharpe: {state_stats[i]['Sharpe Ratio']:.2f})")

regime_labels = regime_decoded.map(label_map)
regime_labels.name = 'regime_label'

print("\nRegime distribution:")
for lbl in REGIME_ORDER:
    n = (regime_labels == lbl).sum()
    print(f"  {lbl:<12}: {n:,} days ({n/len(regime_labels):.1%})")
print(f"\nregime_labels ready: {len(regime_labels):,} observations")

In [ ]:
regime_change = regime_labels.astype(str) != regime_labels.astype(str).shift(1)
spell_id = regime_change.cumsum()

spells = []
for sid in spell_id.unique():
    mask = spell_id == sid
    regime_in_spell = regime_labels[mask].iloc[0]
    start_date = regime_labels[mask].index[0]
    end_date   = regime_labels[mask].index[-1]
    length_days = (end_date - start_date).days + 1
    spells.append({
        "Regime": regime_in_spell,
        "Start":  start_date.strftime("%Y-%m-%d"),
        "End":    end_date.strftime("%Y-%m-%d"),
        "Duration (days)": length_days,
    })

spells_df = pd.DataFrame(spells)

print("=== REGIME DURATION ANALYSIS ===")
for regime in REGIME_ORDER:
    ds = spells_df[spells_df["Regime"] == regime]["Duration (days)"]
    if len(ds) == 0:
        print(f"{regime}: no spells found")
        continue
    print(f"{regime}:")
    print(f"  Mean: {ds.mean():.0f}d (~{ds.mean()/7:.1f}wk)  Median: {ds.median():.0f}d  N spells: {len(ds)}")

stability_ok = all(
    spells_df[spells_df["Regime"] == r]["Duration (days)"].mean() >= 7
    for r in REGIME_ORDER if len(spells_df[spells_df["Regime"] == r]) > 0
)
print(f"\nTemporal Stability (mean >= 7d per regime): {'PASS' if stability_ok else 'WARN -- regimes may be noise-following'}")

In [ ]:
from itertools import permutations as _perms

X = bench_aligned.values.reshape(-1, 1)
baseline_states = model.predict(X)
all_perms = list(_perms([0, 1, 2]))

agreement_fractions = []
seed_results = []

for seed in range(1, 10):
    m = GaussianMixture(n_components=3, covariance_type='diag', max_iter=100, tol=1e-4, random_state=seed)
    m.fit(X)
    states_seed = m.predict(X)
    best = max(
        np.mean(np.array([p[s] for s in states_seed]) == baseline_states)
        for p in all_perms
    )
    agreement_fractions.append(best)
    seed_results.append({'seed': seed, 'agreement': best, 'status': 'pass' if best >= 0.90 else 'FAIL'})

stable_seeds = sum(1 for a in agreement_fractions if a >= 0.90)
print("=== SEED STABILITY CHECK ===")
print("Baseline: seed 0  |  Comparison: seeds 1-9  |  Permutation-aware (3! = 6)")
for r in seed_results:
    print(f"  Seed {r['seed']}: {r['agreement']:.1%}  [{r['status']}]")
print(f"\nSeed stability: {stable_seeds}/9 seeds >=90% agreement.")
print("Status: STABLE" if stable_seeds >= 8 else "Status: UNSTABLE -- review fit")

In [ ]:
posteriors = model.predict_proba(X)

risk_on_state_idx = next(k for k, v in label_map.items() if v == 'Risk-On')

regime_posteriors = pd.Series(posteriors[:, risk_on_state_idx], index=bench_aligned.index)
regime_posteriors.name = 'p_risk_on'

print("=== POSTERIOR PROBABILITY: P(Risk-On) ===")
print(f"  Risk-On state index:        {risk_on_state_idx}")
print(f"  Mean P(Risk-On):            {regime_posteriors.mean():.2%}")
print(f"  Min:                        {regime_posteriors.min():.2%}")
print(f"  Max:                        {regime_posteriors.max():.2%}")
print(f"  Days with P(Risk-On) > 0.5: {(regime_posteriors > 0.5).sum()} ({(regime_posteriors > 0.5).mean():.1%})")
print(f"\nregime_posteriors ready: {len(regime_posteriors):,} observations")

In [ ]:
# Single line: P(Risk-On) over full date range with 0.5 decision boundary (Decision D-03)
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(regime_posteriors.index, regime_posteriors.values,
        label='P(Risk-On)', linewidth=1.5, color=COLOR_BENCH)

# Decision boundary at 0.5 (dashed)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7,
           label='Decision boundary (0.5)')

ax.set_xlabel('Date')
ax.set_ylabel('Posterior Probability')
ax.set_title('Regime Posterior Probability: P(Risk-On)')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=== PHASE 2 OUTPUTS ===")
print(f"\nregime_labels")
print(f"  Type:   {type(regime_labels).__name__}")
print(f"  Index:  {regime_labels.index.dtype} ({regime_labels.index[0].date()} to {regime_labels.index[-1].date()})")
print(f"  Values: {sorted(regime_labels.unique())}")
print(f"  Counts: {regime_labels.value_counts().to_dict()}")

print(f"\nregime_posteriors")
print(f"  Type:   {type(regime_posteriors).__name__}")
print(f"  Range:  [{regime_posteriors.min():.4f}, {regime_posteriors.max():.4f}]")
print(f"  NaN:    {regime_posteriors.isnull().sum()}")

assert set(regime_labels.unique()) == {'Risk-On', 'Transition', 'Risk-Off'}, "regime_labels must contain exactly 3 states"
assert regime_posteriors.isnull().sum() == 0, "regime_posteriors must not contain NaN"
assert (regime_posteriors >= 0).all() and (regime_posteriors <= 1).all(), "regime_posteriors must be in [0, 1]"
assert regime_labels.index.equals(regime_posteriors.index), "index mismatch"

print("\nPhase 2 complete. All output variables validated.")
print("Next: Phase 3 (Fund Analysis) and Phase 4 (Regime Stability & Transitions).")

## 3. Fund Analysis by Regime

Regime-conditional statistics for the fund: return, volatility, Sharpe, and max drawdown
computed separately for Risk-On and Risk-Off periods. Alpha and beta estimated via OLS per regime.
The summary table is the core deliverable for this phase.

In [ ]:
# ------------------------------------------------------------------
# Section 3.1: Regime-Conditional Fund Statistics (FUND-01, FUND-03)
# ------------------------------------------------------------------

fund_stats_full = compute_summary_stats(fund_aligned)

regime_fund_stats = {}
for label in REGIME_ORDER:
    mask = regime_labels == label
    regime_fund_stats[label] = compute_summary_stats(fund_aligned[mask])

print(f"=== REGIME-CONDITIONAL FUND STATISTICS ({FUND_SYMBOL}) ===")
print(f"Full history: n={fund_stats_full['Observations']:,}  ({fund_stats_full['Start Date']} to {fund_stats_full['End Date']})")
print(f"  Annual Return:     {fund_stats_full['Annual Return']:.2%}")
print(f"  Annual Volatility: {fund_stats_full['Annual Volatility']:.2%}")
print(f"  Sharpe Ratio:      {fund_stats_full['Sharpe Ratio']:.2f}")
print(f"  Max Drawdown:      {fund_stats_full['Max Drawdown']:.2%}")

for label, stats in regime_fund_stats.items():
    print(f"\n{label}: n={stats['Observations']:,}  ({stats['Start Date']} to {stats['End Date']})")
    print(f"  Annual Return:     {stats['Annual Return']:.2%}")
    print(f"  Annual Volatility: {stats['Annual Volatility']:.2%}")
    print(f"  Sharpe Ratio:      {stats['Sharpe Ratio']:.2f}")
    print(f"  Max Drawdown:      {stats['Max Drawdown']:.2%}")

In [ ]:
# ------------------------------------------------------------------
# Section 3.2: Alpha and Beta by Regime via OLS (FUND-02, FUND-03)
# ------------------------------------------------------------------

import statsmodels.api as sm

ols_results = {}

for label in REGIME_ORDER:
    mask    = regime_labels == label
    fund_r  = fund_aligned[mask]
    bench_r = bench_aligned[mask]
    assert fund_r.index.equals(bench_r.index), f"Index mismatch for {label}"

    X_reg = sm.add_constant(bench_r, has_constant='add')
    X_reg.columns = ['const', 'bench']
    fit = sm.OLS(fund_r, X_reg).fit()

    ols_results[label] = {
        'alpha_annual': (1 + fit.params['const']) ** 252 - 1,
        'beta':         fit.params['bench'],
        'r_squared':    fit.rsquared,
        'alpha_pval':   fit.pvalues['const'],
        'beta_pval':    fit.pvalues['bench'],
        'n':            len(fund_r),
        'start':        fund_r.index.min().strftime('%Y-%m-%d'),
        'end':          fund_r.index.max().strftime('%Y-%m-%d'),
    }

print(f"=== ALPHA / BETA BY REGIME -- OLS ({FUND_SYMBOL} ~ {BENCHMARK_TICKER}) ===")
for label, r in ols_results.items():
    sig = '  *' if r['alpha_pval'] < 0.05 else ''
    print(f"\n{label}  (n={r['n']:,}, {r['start']} to {r['end']}):")
    print(f"  Alpha (ann.): {r['alpha_annual']:+.2%}{sig}  (p={r['alpha_pval']:.3f})")
    print(f"  Beta:          {r['beta']:.3f}  (p={r['beta_pval']:.3f})")
    print(f"  R2:            {r['r_squared']:.3f}")
print("\n* p < 0.05")

In [ ]:
# ------------------------------------------------------------------
# Section 3.3: Summary Statistics Table -- Core Deliverable (FUND-04)
# ------------------------------------------------------------------

def pct(v): return f"{v:.1%}"
def dec(v): return f"{v:.2f}"
def nn(v):  return f"{int(v):,}"

rows = {
    'Annual Return':     {'Full History': pct(fund_stats_full['Annual Return']),
                          **{lbl: pct(regime_fund_stats[lbl]['Annual Return']) for lbl in REGIME_ORDER}},
    'Annual Volatility': {'Full History': pct(fund_stats_full['Annual Volatility']),
                          **{lbl: pct(regime_fund_stats[lbl]['Annual Volatility']) for lbl in REGIME_ORDER}},
    'Sharpe Ratio':      {'Full History': dec(fund_stats_full['Sharpe Ratio']),
                          **{lbl: dec(regime_fund_stats[lbl]['Sharpe Ratio']) for lbl in REGIME_ORDER}},
    'Max Drawdown':      {'Full History': pct(fund_stats_full['Max Drawdown']),
                          **{lbl: pct(regime_fund_stats[lbl]['Max Drawdown']) for lbl in REGIME_ORDER}},
    'Alpha (ann.)':      {'Full History': '--',
                          **{lbl: pct(ols_results[lbl]['alpha_annual']) for lbl in REGIME_ORDER}},
    'Beta':              {'Full History': '--',
                          **{lbl: dec(ols_results[lbl]['beta']) for lbl in REGIME_ORDER}},
    'R2':                {'Full History': '--',
                          **{lbl: dec(ols_results[lbl]['r_squared']) for lbl in REGIME_ORDER}},
    'Observations (n)':  {'Full History': nn(fund_stats_full['Observations']),
                          **{lbl: nn(regime_fund_stats[lbl]['Observations']) for lbl in REGIME_ORDER}},
}

cols = ['Full History'] + REGIME_ORDER
summary_df = pd.DataFrame(rows).T[cols]

print(f'=== {FUND_SYMBOL}: FUND BEHAVIOR BY REGIME ===')
print(f'Benchmark: {BENCHMARK_TICKER}')
print(f'Period:    {fund_aligned.index.min().date()} to {fund_aligned.index.max().date()}')
print(f'Model:     3-state Gaussian Mixture on benchmark returns')
print()
print(summary_df.to_string())
print()
print('Notes:')
print('  Alpha estimated via OLS per regime; annualized.')
print('  Sharpe uses 2% annual risk-free rate. Max drawdown is regime-conditional.')

## 4. Regime Stability & Transitions

Transition probability matrix and expected durations derived from the empirical state sequence. Current regime read-out as of the latest data date.

In [ ]:
# ------------------------------------------------------------------
# Section 4.1: Regime Transition Matrix (STAB-01)
# Computed from regime_labels sequence (labeled, not integer states)
# Rows = from-state, cols = to-state
# ------------------------------------------------------------------

STATES = REGIME_ORDER

# Count transitions
trans_counts = pd.DataFrame(0, index=STATES, columns=STATES, dtype=int)
for s1, s2 in zip(regime_labels.values[:-1], regime_labels.values[1:]):
    trans_counts.loc[s1, s2] += 1

# Row-normalize -> probabilities
trans_probs = trans_counts.div(trans_counts.sum(axis=1), axis=0)

print("=== REGIME TRANSITION PROBABILITY MATRIX ===")
for s in STATES:
    row = "  ".join(f"{trans_probs.loc[s, t]:.3f}" for t in STATES)
    print(f"  {s:<10} -> {row}")
print(f"Column headers: {STATES}")
print("Transition counts:")
print(trans_counts.to_string())

# Heatmap (pure matplotlib -- seaborn unavailable on server)
data = trans_probs.values
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(data, cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Probability")
ax.set_xticks(range(len(STATES)))
ax.set_yticks(range(len(STATES)))
ax.set_xticklabels(STATES)
ax.set_yticklabels(STATES)
ax.set_xlabel("To Regime")
ax.set_ylabel("From Regime")
ax.set_title("Regime Transition Probability Matrix")
for i in range(len(STATES)):
    for j in range(len(STATES)):
        txt_color = "white" if data[i, j] > 0.6 else "black"
        ax.text(j, i, f"{data[i, j]:.1%}", ha="center", va="center",
                fontsize=12, color=txt_color, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Section 4.2: Expected Durations + Current Regime Read-Out (STAB-02, STAB-03)
# E[duration] = 1 / (1 - p_ii)  -- geometric distribution assumption
# Duration unit: trading days (the observation frequency).
# Note: GMM has no temporal constraints -- short Risk-Off E[duration]
# reflects empirical fragmentation, not regime quality.
# ------------------------------------------------------------------

print("=== EXPECTED REGIME DURATIONS (STAB-02) ===")
for regime in STATES:
    p_stay = trans_probs.loc[regime, regime]
    if p_stay < 1.0:
        exp_dur = 1.0 / (1.0 - p_stay)
        print(f"  {regime:<10}  p(stay) = {p_stay:.3f}  ->  E[duration] = {exp_dur:.1f} trading days (~{exp_dur / 5:.1f} wk)")
    else:
        print(f"  {regime:<10}  p(stay) = 1.000  ->  absorbing state")

print()
print("  Note: GMM has no temporal smoothing. Short Risk-Off E[duration]")
print("  reflects fragmented single-day assignments; treat with caution.")

# Current regime read-out (STAB-03)
current_label     = regime_labels.iloc[-1]
current_date      = regime_labels.index[-1]
current_posterior = regime_posteriors.iloc[-1]

# Consecutive trading days at end of series in same regime
streak = int((regime_labels.iloc[::-1] == current_label).cumprod().sum())

print(f"
=== CURRENT REGIME READ-OUT (STAB-03) ===")
print(f"  As of:          {current_date.strftime('%Y-%m-%d')}")
print(f"  Regime:         {current_label}")
print(f"  P(Risk-On):     {current_posterior:.2%}")
print(f"  Days in spell:  {streak} trading days (~{streak / 5:.1f} wk)")

print(f"
=== REGIME FREQUENCY (full history) ===")
for regime in STATES:
    n   = (regime_labels == regime).sum()
    pct = n / len(regime_labels)
    print(f"  {regime:<10}  {n:,} days  ({pct:.1%} of history)")

print(f"
Phase 4 complete.")